## Deployment: Aplikasi Prediksi Harga Cabai Merah Besar dengan Streamlit

Notebook ini merupakan tahap **Deployment** dalam metodologi **CRISP-DM**, yang bertujuan untuk membangun antarmuka web interaktif menggunakan **Streamlit** sehingga pengguna non-teknis dapat melakukan prediksi harga cabai secara real-time.

Model yang digunakan adalah **Linear Regression** terbaik yang telah disimpan pada tahap evaluasi sebelumnya (`model_cabai_lr.pkl`).

Antarmuka dilengkapi dengan **tombol preset** untuk skenario cuaca (kemarau, hujan, transisi) dan skenario harga historis (rendah, normal, tinggi, sangat tinggi).

> **Catatan**: Streamlit tidak dapat dijalankan secara *inline* di notebook seperti Gradio. Notebook ini menguji seluruh logic, kemudian menjalankan script `scripts/app_streamlit.py` sebagai background process via `subprocess.Popen`. Layout dan fungsi identik dengan notebook `Deployment_Gradio.ipynb`.

### 1. Instalasi Dependency

Pastikan pustaka `streamlit` telah terinstal. Jika belum, jalankan cell berikut.

In [ ]:
# !pip install streamlit joblib scikit-learn pandas numpy

### 2. Import Library

In [1]:
import os
import joblib
import pandas as pd
import numpy as np
import streamlit as st
from pathlib import Path

### 3. Memuat Model Linear Regression

Model terbaik (`model_cabai_lr.pkl`) dimuat dari folder `models/`. Path dibuat fleksibel agar notebook dapat dijalankan dari berbagai lokasi.

> **Catatan Streamlit**: Di `app_streamlit.py`, model loading menggunakan `@st.cache_resource` untuk menghindari reload setiap rerun. Di notebook ini, model dimuat secara biasa untuk pengujian.

In [2]:
# Path fleksibel untuk mencari file model
possible_model_paths = [
    Path('../models/model_cabai_lr.pkl'),
    Path('model_cabai_lr.pkl'),
    Path('models/model_cabai_lr.pkl'),
]

model_path = None
for path in possible_model_paths:
    if path.exists():
        model_path = path
        print(f"Model ditemukan di: {path}")
        break

if model_path is None:
    raise FileNotFoundError("File model_cabai_lr.pkl tidak ditemukan. Pastikan file berada di folder models/.")

model = joblib.load(model_path)
print(f"Model berhasil dimuat: {type(model).__name__}")

Model ditemukan di: ..\models\model_cabai_lr.pkl
Model berhasil dimuat: LinearRegression


c:\Users\jeebr\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator LinearRegression from version 1.8.0 when using version 1.4.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


### 4. Pemetaan Bulan dan Fungsi Prediksi

Pada antarmuka Streamlit, bulan ditampilkan dalam format nama bulan Indonesia (Januari–Desember), lalu dikonversi ke integer (1–12) sebelum dikirim ke model.

Fungsi `prediksi_harga_cabai` menerima 6 input dari UI, menyusunnya ke DataFrame dengan kolom yang sesuai dengan `X_train`, melakukan prediksi, dan memformat output ke Rupiah.

In [3]:
# Pemetaan nama bulan Indonesia ke integer
BULAN_MAP = {
    "Januari": 1, "Februari": 2, "Maret": 3, "April": 4, "Mei": 5, "Juni": 6,
    "Juli": 7, "Agustus": 8, "September": 9, "Oktober": 10, "November": 11, "Desember": 12
}

# Urutan kolom fitur yang persis sama dengan X_train
FEATURE_COLUMNS = ['Cabai_lag_1', 'Cabai_lag_7', 'RR_lag_45', 'RH_lag_30', 'RR_rolling_mean_14', 'bulan']


def prediksi_harga_cabai(cabai_lag_1, cabai_lag_7, rr_lag_45, rh_lag_30, rr_rolling_mean_14, nama_bulan):
    try:
        bulan_val = BULAN_MAP.get(nama_bulan, 1)

        input_data = pd.DataFrame({
            'Cabai_lag_1': [float(cabai_lag_1)],
            'Cabai_lag_7': [float(cabai_lag_7)],
            'RR_lag_45': [float(rr_lag_45)],
            'RH_lag_30': [float(rh_lag_30)],
            'RR_rolling_mean_14': [float(rr_rolling_mean_14)],
            'bulan': [int(bulan_val)]
        })

        predicted_value = model.predict(input_data)[0]
        predicted_value = max(0.0, predicted_value)

        return f"Rp {predicted_value:,.2f}"

    except Exception as err:
        return f"Terjadi kesalahan saat memproses data: {str(err)}"

### 4.1. Preset Skenario Cuaca dan Harga Historis

Tombol preset memungkinkan pengguna mengisi seluruh field input secara otomatis hanya dengan satu klik:

- **Preset Cuaca**: mewakili kondisi kemarau panas, kemarau sejuk, musim transisi, hujan ringan, dan hujan lebat di Bandung.
- **Preset Harga**: mewakili harga cabai pada level rendah (Q10), normal (median), tinggi (Q75), dan sangat tinggi (Q90+).

> **Catatan Streamlit**: Di Streamlit, preset buttons menggunakan pola `st.session_state` flag. Ketika tombol diklik → set flag → `st.rerun()` → flag detected → inject values ke session_state → pop flag → widgets render dengan nilai baru.

In [4]:
# --- PRESET CUACA BANDUNG ---
PRESET_CUACA = {
    "\u2600\ufe0f Kemarau Panas": {
        "rr_lag_45": 0.0,
        "rh_lag_30": 65,
        "rr_rolling_mean_14": 0.0,
        "nama_bulan": "September",
    },
    "\ud83c\udf24\ufe0f Kemarau Sejuk": {
        "rr_lag_45": 0.0,
        "rh_lag_30": 72,
        "rr_rolling_mean_14": 1.5,
        "nama_bulan": "Agustus",
    },
    "\ud83c\udf25\ufe0f Musim Transisi": {
        "rr_lag_45": 3.0,
        "rh_lag_30": 78,
        "rr_rolling_mean_14": 5.0,
        "nama_bulan": "Oktober",
    },
    "\ud83c\udf27\ufe0f Hujan Ringan": {
        "rr_lag_45": 8.0,
        "rh_lag_30": 82,
        "rr_rolling_mean_14": 7.5,
        "nama_bulan": "November",
    },
    "\u26c8\ufe0f Hujan Lebat": {
        "rr_lag_45": 35.0,
        "rh_lag_30": 88,
        "rr_rolling_mean_14": 15.0,
        "nama_bulan": "Januari",
    },
}

# --- PRESET HARGA CABAI HISTORIS ---
PRESET_HARGA = {
    "\ud83d\udcc9 Harga Rendah": {
        "cabai_lag_1": 42500,
        "cabai_lag_7": 42500,
        "nama_bulan": "November",
    },
    "\ud83d\udcca Harga Normal": {
        "cabai_lag_1": 53500,
        "cabai_lag_7": 53500,
        "nama_bulan": "September",
    },
    "\ud83d\udcc8 Harga Tinggi": {
        "cabai_lag_1": 65500,
        "cabai_lag_7": 65550,
        "nama_bulan": "Juni",
    },
    "\ud83d\udd25 Harga Sangat Tinggi": {
        "cabai_lag_1": 71400,
        "cabai_lag_7": 71400,
        "nama_bulan": "Maret",
    },
}

print("Preset cuaca:")
for k, v in PRESET_CUACA.items():
    print(f"  {k}: RR_lag_45={v['rr_lag_45']}, RH_lag_30={v['rh_lag_30']}, RR_roll={v['rr_rolling_mean_14']}, Bulan={v['nama_bulan']}")

print("\nPreset harga:")
for k, v in PRESET_HARGA.items():
    print(f"  {k}: Cabai_lag_1={v['cabai_lag_1']}, Cabai_lag_7={v['cabai_lag_7']}, Bulan={v['nama_bulan']}")

Exception in callback BaseAsyncIOLoop._handle_events(1564, 1)
handle: <Handle BaseAsyncIOLoop._handle_events(1564, 1)>
UnicodeEncodeError: 'utf-8' codec can't encode characters in position 94-95: surrogates not allowed

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "C:\Users\jeebr\AppData\Roaming\Python\Python312\site-packages\jupyter_client\session.py", line 143, in orjson_packer
    return orjson.dumps(obj, default=json_default, option=option)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: str is not valid UTF-8: surrogates not allowed

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "C:\Users\jeebr\AppData\Roaming\Python\Python312\site-packages\jupyter_client\session.py", line 103, in json_packer
    ).encode("utf8", errors="surrogateescape")
      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'utf-8' codec can'

### 5. Pengujan Fungsi Prediksi

Sebelum membangun UI, kita uji fungsi prediksi dengan data contoh untuk memastikan model dan fungsi bekerja dengan benar.

In [5]:
# Uji fungsi prediksi dengan data contoh (sama seperti pada notebook modelling)
hasil_test = prediksi_harga_cabai(
    cabai_lag_1=70000,
    cabai_lag_7=68000,
    rr_lag_45=10,
    rh_lag_30=82,
    rr_rolling_mean_14=7,
    nama_bulan="Juni"
)

print(f"Hasil prediksi uji coba: {hasil_test}")

Hasil prediksi uji coba: Rp 69,890.85


### 6. Desain Antarmuka Streamlit

Antarmuka Streamlit dirancang dengan layout **wide** dan konfigurasi halaman khusus. Layout terdiri dari:
- **Baris preset harga**: 4 tombol dalam `st.columns(4)` untuk mengisi harga cabai historis
- **Baris preset cuaca**: 5 tombol dalam `st.columns(5)` untuk mengisi variabel iklim Bandung
- **Kolom kiri**: Parameter harga cabai historis (`st.number_input` + `st.selectbox`)
- **Kolom kanan**: Parameter iklim Bandung (`st.number_input` + `st.slider`)
- **Output**: `st.success()` box dengan format Rupiah

Script lengkap `scripts/app_streamlit.py` telah dibuat secara terpisah. Cell berikut memverifikasi bahwa script valid dan siap dijalankan.

In [6]:
# Verifikasi script app_streamlit.py
SCRIPT_PATH = Path('../scripts/app_streamlit.py')

if not SCRIPT_PATH.exists():
    # Fallback paths
    for p in [Path('scripts/app_streamlit.py'), Path('app_streamlit.py')]:
        if p.exists():
            SCRIPT_PATH = p
            break

if SCRIPT_PATH.exists():
    with open(SCRIPT_PATH, 'r', encoding='utf-8') as f:
        content = f.read()
    print(f"Script ditemukan: {SCRIPT_PATH}")
    print(f"Jumlah baris: {len(content.splitlines())}")
    print(f"Jumlah karakter: {len(content)}")

    # Verifikasi key components
    checks = [
        ('st.set_page_config', 'Page config'),
        ('@st.cache_resource', 'Model caching'),
        ('BULAN_MAP', 'Bulan mapping'),
        ('PRESET_CUACA', 'Preset cuaca'),
        ('PRESET_HARGA', 'Preset harga'),
        ('preset_harga', 'Preset handler harga'),
        ('preset_cuaca', 'Preset handler cuaca'),
        ('prediksi_harga_cabai', 'Fungsi prediksi'),
        ('st.columns', 'Layout columns'),
        ('st.success', 'Output display'),
    ]
    all_ok = True
    for keyword, label in checks:
        found = keyword in content
        status = '\u2713' if found else '\u2717'
        print(f"  {status} {label}: {'found' if found else 'NOT FOUND'}")
        if not found:
            all_ok = False

    if all_ok:
        print("\n\u2705 Script valid dan siap dijalankan.")
    else:
        print("\n\u274c Script tidak lengkap. Periksa file app_streamlit.py.")
else:
    print("Script app_streamlit.py tidak ditemukan.")
    print("Pastikan file berada di folder scripts/.")

Script ditemukan: ..\scripts\app_streamlit.py
Jumlah baris: 191
Jumlah karakter: 6885
  ✓ Page config: found
  ✓ Model caching: found
  ✓ Bulan mapping: found
  ✓ Preset cuaca: found
  ✓ Preset harga: found
  ✓ Preset handler harga: found
  ✓ Preset handler cuaca: found
  ✓ Fungsi prediksi: found
  ✓ Layout columns: found
  ✓ Output display: found

✅ Script valid dan siap dijalankan.


### 7. Jalankan Aplikasi Streamlit

Streamlit dijalankan sebagai **background process** via `subprocess.Popen`. Aplikasi akan tersedia di `http://localhost:8501`.

> Streamlit berjalan sebagai server yang terus aktif — notebook cell ini akan launch server dan selesai, server tetap berjalan di background.

In [7]:
import subprocess
import time

# Kill existing Streamlit process on port 8501 if any
try:
    result = subprocess.run(
        ['netstat', '-ano'], capture_output=True, text=True
    )
    for line in result.stdout.splitlines():
        if ':8501' in line and 'LISTENING' in line:
            pid = line.strip().split()[-1]
            subprocess.run(['taskkill', '/F', '/PID', pid], capture_output=True)
            print(f"Killed existing process on port 8501 (PID: {pid})")
except Exception:
    pass

# Launch Streamlit as background process
proc = subprocess.Popen(
    ['streamlit', 'run', str(SCRIPT_PATH), '--server.headless', 'true', '--server.port', '8501'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

time.sleep(3)
print(f"Streamlit running at http://localhost:8501 (PID: {proc.pid})")
print("Buka browser dan akses URL tersebut.")

Streamlit running at http://localhost:8501 (PID: 6224)
Buka browser dan akses URL tersebut.
